### Notebook 06: Validación externa- Generalización del modelo al dataset Extended

In [5]:
# --- Cargar del dataset Extended para ver su estructura ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)

RUTA_BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
df_ext = pd.read_csv(os.path.join(RUTA_BASE, 'data', 'raw', 'general_data.csv'))

print("Forma:", df_ext.shape)
print("\nColumnas del Extended:")
print(df_ext.columns.tolist())
print("\n¿Tiene columna Attrition?:", 'Attrition' in df_ext.columns)

Forma: (4410, 24)

Columnas del Extended:
['Age', 'Attrition', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeID', 'Gender', 'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'Over18', 'PercentSalaryHike', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

¿Tiene columna Attrition?: True


In [7]:
# --- Carga del IBM HR limpio para reconstruir X_train ---
from sklearn.model_selection import train_test_split

# Carga del dataset limpio
ibm_clean = pd.read_csv(os.path.join(RUTA_BASE, 'data', 'processed', 'ibm_hr_clean.csv'))

# one-hot encoding usado en el notebook 4
ibm_encoded = pd.get_dummies(ibm_clean, drop_first=True)
X = ibm_encoded.drop(columns=['Attrition'])
y = ibm_encoded['Attrition']

# Split usado en el 4 (test_size=0.3, random_state=42, sin stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

print(f'Train: {X_train.shape[0]} filas | {X_train.shape[1]} columnas')
print(f'Test: {X_test.shape[0]} filas')

Train: 1029 filas | 44 columnas
Test: 441 filas


In [9]:
# Compración de features del modelo vs el Extended
features_modelo = X_train.columns.tolist()   # Las 44 que columnas que espera el modelo

cols_extended = df_ext.columns.tolist()

# Comparamos por nombre base
print("Columnas del Extended:", len(cols_extended))
print("\nEl modelo espera estas variables base. ¿Están en el Extended?")
for col in ['OverTime', 'JobSatisfaction', 'EnvironmentSatisfaction',
            'WorkLifeBalance', 'YearsInCurrentRole', 'JobInvolvement']:
    print(f"  {col}: {'SÍ' if col in cols_extended else 'No, falta'}")

Columnas del Extended: 24

El modelo espera estas variables base. ¿Están en el Extended?
  OverTime: No, falta
  JobSatisfaction: No, falta
  EnvironmentSatisfaction: No, falta
  WorkLifeBalance: No, falta
  YearsInCurrentRole: No, falta
  JobInvolvement: No, falta


## Conclusión: incompatibilidad de esquemas y decisión metodológica

Se planteó validar externamente el modelo con el dataset Extended ( que tiene 4.410 registros) para evaluar su generalización a una población distinta
a la del IBM HR usado en el entrenamiento.

El análisis de compatibilidad de variables reveló que el dataset Extended carece de las variables más predictivas del modelo ganador: 
OverTime, JobSatisfaction, EnvironmentSatisfaction, WorkLifeBalance y JobInvolvement; centrales en el marco teórico JD-R sobre el que se sustenta el 
estudio. También faltan YearsInCurrentRole y otras variables incluidas en el modelo final.

Dado que el modelo espera 44 variables y el Extended no aporta un subconjunto que preserve los predictores clave, una validación externa directa no es
viable y forzarla evaluaría un modelo distinto al presentado, produciendo métricas no representativas.

Se descartó la alternativa de reentrenar un modelo reducido a variables comunes, por dos razones: 
- Contradice el marco JD-R al eliminar las dimensiones de satisfacción y la carga laboral (OverTime).
- No validaría el modelo final, sino uno diferente.

Por lo que la decisión es documentar la incompatibilidad de esquemas, que según lo validado es algo común entre datasets públicos de People Analytics,
como limitación del estudio. La robustez del modelo se sustenta en la validación cruzada estratificada de 5 particiones, realizada en el Notebook 4. 
La validación en datos organizacionales con esquema homogéneo se propone como línea de trabajo futuro.